In [1]:
import pandas as pd
import numpy as np

In [2]:
staked_size = pd.read_csv('../int/staked_pool_size.csv')
staked_category = pd.read_csv('../int/staked_category.csv')
staked_pool = pd.read_csv('../int/staked_pool.csv')
validator_index = pd.read_csv('../data/validator_metadata.csv', usecols=['validator_pubkey', 'validator_index'])
validator_metadata = pd.read_csv('../int/validator_metadata_v2.csv', usecols=['validator_index', 'pool', 'category', 'pool_size_label'])
largest_pools = ('Lido','Coinbase','Binance','Rocketpool','Kraken','OKX','Bitcoin Suisse','Ledger Live','Ether.Fi','Mantle')
validator_metadata.loc[~validator_metadata['pool'].isin(largest_pools), 'pool'] = 'Other Stakers'


In [4]:
deposits = pd.read_csv('../data/deposits_v2.csv')
deposits = deposits.sort_values(by='slot').reset_index(drop=True)
deposits['validator_pubkey'] = '0' + deposits['validator_pubkey'].str.replace('\\', '')

# Fetch and transform deposits
deposits = pd.merge(deposits, validator_index, on='validator_pubkey', how='left')
deposits = deposits.drop(columns=('validator_pubkey'))

# Merge validator metadata
deposits = pd.merge(deposits, validator_metadata, on='validator_index', how='left')
deposits = deposits[deposits['pool'].notna()]
deposits = deposits.drop(columns='validator_index')
deposits['amount'] = deposits['amount'] / (10**9)
deposits

,slot,amount,pool,category,pool_size_label
0,1229,32.0,Other Stakers,Unidentified,100+
1,1229,32.0,Other Stakers,Unidentified,6-19
2,1229,32.0,Other Stakers,Unidentified,100+
3,1229,32.0,Other Stakers,Unidentified,1
4,1229,32.0,Other Stakers,Unidentified,6-19
...,...,...,...,...,...
1471671,9167900,32.0,Other Stakers,CEX,100+
1471673,9167900,32.0,Other Stakers,CEX,100+
1471674,9167900,32.0,Other Stakers,CEX,100+
1471675,9167900,32.0,Other Stakers,CEX,100+


In [6]:
deposits['slot'] = ((deposits['slot'] - 1200) // 300) * 300 + 1200
deposits = deposits[deposits['slot'] <= 8986176]
deposits

,slot,amount,pool,category,pool_size_label
0,1200,32.0,Other Stakers,Unidentified,100+
1,1200,32.0,Other Stakers,Unidentified,6-19
2,1200,32.0,Other Stakers,Unidentified,100+
3,1200,32.0,Other Stakers,Unidentified,1
4,1200,32.0,Other Stakers,Unidentified,6-19
...,...,...,...,...,...
1444152,8985600,32.0,Other Stakers,CEX,100+
1444153,8985600,32.0,Other Stakers,CEX,100+
1444154,8985600,32.0,Other Stakers,Unidentified,1
1444155,8985600,32.0,Other Stakers,Liquid Restaking,100+


In [10]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['slot'] = ((slots_df['slot'] - 1200) // 300) * 300 + 1200
slots_df['pool_size_label'] = np.nan
slots_df['amount'] = np.nan

# Get unique pool size labels
columns = deposits['pool_size_label'].unique()

# Assign pool size labels cyclically
slots_df['pool_size_label'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Group deposits by slot and pool_size_label, and sum the number of exits
deposits_grouped = deposits.groupby(['slot', 'pool_size_label'])['amount'].sum().reset_index()

# Create the pivot table with the grouped and summed exits
deposits_size_pivot = deposits_grouped.pivot_table(index='slot', columns='pool_size_label', values='amount', fill_value=0)

# Merge the pivot table with slots_df
slots_pivot = slots_df.pivot_table(index='slot', columns='pool_size_label', values='amount', dropna=False)
deposits_size = slots_pivot.combine_first(deposits_size_pivot).fillna(0)

deposits_size['total'] = deposits_size.sum(axis=1)

# Display the pivot table
deposits_size

pool_size_label,1,100+,2-5,20-99,6-19,total
slot,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0
300,0.0,0.0,0.0,0.0,0.0,0.0
600,0.0,0.0,0.0,0.0,0.0,0.0
900,0.0,0.0,0.0,0.0,0.0,0.0
1200,6912.0,83520.0,7840.0,17440.0,9216.0,124928.0
...,...,...,...,...,...,...
8984700,0.0,0.0,0.0,0.0,0.0,0.0
8985000,0.0,0.0,0.0,0.0,0.0,0.0
8985300,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['slot'] = ((slots_df['slot'] - 1200) // 300) * 300 + 1200
slots_df['category'] = np.nan
slots_df['amount'] = np.nan

# Get unique pool size labels
columns = deposits['category'].unique()

# Assign pool size labels cyclically
slots_df['category'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Group deposits by slot and category, and sum the number of exits
deposits_grouped = deposits.groupby(['slot', 'category'])['amount'].sum().reset_index()

# Create the pivot table with the grouped and summed exits
deposits_category_pivot = deposits_grouped.pivot_table(index='slot', columns='category', values='amount', fill_value=0)

# Merge the pivot table with slots_df
slots_pivot = slots_df.pivot_table(index='slot', columns='category', values='amount', dropna=False)
deposits_category = slots_pivot.combine_first(deposits_category_pivot).fillna(0)

deposits_category['total'] = deposits_category.sum(axis=1)

# Display the pivot table
deposits_category

category,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total
slot,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
300,0.0,0.0,0.0,0.0,0.0,0.0,0.0
600,0.0,0.0,0.0,0.0,0.0,0.0,0.0
900,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1200,38368.0,0.0,3456.0,9248.0,1984.0,71872.0,124928.0
...,...,...,...,...,...,...,...
8984700,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8985000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8985300,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['slot'] = ((slots_df['slot'] - 1200) // 300) * 300 + 1200
slots_df['pool'] = np.nan
slots_df['amount'] = np.nan

# Get unique pool size labels
columns = deposits['pool'].unique()

# Assign pool size labels cyclically
slots_df['pool'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Group deposits by slot and pool, and sum the number of exits
deposits_grouped = deposits.groupby(['slot', 'pool'])['amount'].sum().reset_index()

# Create the pivot table with the grouped and summed exits
deposits_pool_pivot = deposits_grouped.pivot_table(index='slot', columns='pool', values='amount', fill_value=0)

# Merge the pivot table with slots_df
slots_pivot = slots_df.pivot_table(index='slot', columns='pool', values='amount', dropna=False)
deposits_pool = slots_pivot.combine_first(deposits_pool_pivot).fillna(0)

deposits_pool['total'] = deposits_pool.sum(axis=1)

# Display the pivot table
deposits_pool

pool,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
slot,,,,,,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
300,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
600,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
900,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1200,0.0,3936.0,0.0,0.0,0.0,64.0,0.0,0.0,0.0,120928.0,0.0,124928.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8984700,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8985000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8985300,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
deposits_size.to_csv('../int/deposits_size.csv')
deposits_category.to_csv('../int/deposits_category.csv')
deposits_pool.to_csv('../int/deposits_pool.csv')